# Unit 18 / Chapter 18: Open Research Problems in Quantum AI

> **Main Learning Objective:** Understand the biggest open questions holding QML back today: barren plateaus, data loading, and provable speedups. Learn how to read a quantum ML paper productively and pick a problem worth working on.

| Section | Topic |
|---|---|
| 18.1 | Barren plateaus and why gradients vanish |
| 18.2 | The data-loading bottleneck |
| 18.3 | When does QML actually beat classical? |
| 18.4 | Reading QML papers and picking research problems |

---
## Setup

In [ ]:
# Verify libraries. Works in classic Jupyter, JupyterLite/Pyodide, and Colab.
import importlib.util
required = ["numpy", "matplotlib"]
missing = [p for p in required if importlib.util.find_spec(p) is None]
if missing:
    try:
        import piplite
        await piplite.install(missing)
    except ImportError:
        try:
            import micropip
            await micropip.install(missing)
        except ImportError:
            ip = get_ipython()
            ip.run_line_magic('pip', 'install --quiet ' + ' '.join(missing))
import numpy, matplotlib
print("All libraries ready.")

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from IPython.display import HTML, display, Markdown
import math, random
np.random.seed(7); random.seed(7)
plt.rcParams['figure.dpi'] = 100

# Tiny quantum simulator used across all units
def ket0(n):
    s = np.zeros(2**n, dtype=complex); s[0] = 1.0
    return s
def kron_all(mats):
    out = mats[0]
    for m in mats[1:]:
        out = np.kron(out, m)
    return out
I2 = np.eye(2, dtype=complex)
X  = np.array([[0,1],[1,0]], dtype=complex)
Y  = np.array([[0,-1j],[1j,0]], dtype=complex)
Z  = np.array([[1,0],[0,-1]], dtype=complex)
H  = (1/np.sqrt(2))*np.array([[1,1],[1,-1]], dtype=complex)
def Rx(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-1j*s],[-1j*s,c]], dtype=complex)
def Ry(t): c,s = np.cos(t/2), np.sin(t/2); return np.array([[c,-s],[s,c]], dtype=complex)
def Rz(t): return np.array([[np.exp(-1j*t/2),0],[0,np.exp(1j*t/2)]], dtype=complex)
def apply_1q(gate, qubit, n):
    return kron_all([gate if i==qubit else I2 for i in range(n)])
def apply_cnot(control, target, n):
    dim = 2**n
    op = np.zeros((dim, dim), dtype=complex)
    for x in range(dim):
        bits = [(x >> (n-1-i)) & 1 for i in range(n)]
        if bits[control] == 1:
            bits[target] ^= 1
        y = 0
        for b in bits:
            y = (y<<1) | b
        op[y, x] = 1
    return op
def expZ(state, qubit, n):
    Zop = apply_1q(Z, qubit, n)
    return float(np.real(np.conj(state) @ Zop @ state))
print("Quantum simulator ready.")

---
## Course check-in

This logs that you started **Unit 18**. Enter the email you signed up with.

In [ ]:
# ============================================================
# COURSE TRACKING, do not edit
# ============================================================
import json
from urllib.request import Request, urlopen
from urllib.error  import URLError

UNIT_NUMBER = 18
TRACKER_URL = "https://script.google.com/macros/s/AKfycbyp01BDLgzqHk5HbYt7Tl0hYESKo4qRs8AMJsFKUfbNKdbUuzjT6yb1L2qVFd_oz2Ur/exec"

def _post_event(event_type, payload=None):
    body = json.dumps({
        "event_type": event_type,
        "email":      _student_email,
        "unit":       UNIT_NUMBER,
        "payload":    payload or {}
    }).encode("utf-8")
    try:
        req = Request(TRACKER_URL, data=body,
                      headers={"Content-Type": "text/plain;charset=utf-8"})
        urlopen(req, timeout=10).read()
    except URLError as e:
        print("(could not reach tracker:", e, ")")

_student_email = input("Enter the email you signed up with: ").strip().lower()
if "@" not in _student_email:
    raise ValueError("That does not look like a valid email. Re-run this cell.")

print(f"Hi {_student_email}! Logging that you started Unit {UNIT_NUMBER}.")
_post_event("unit_started")

---
# Section 18.1: Barren Plateaus and Why Gradients Vanish

The dirty secret of variational QML: for many circuit designs, the training landscape flattens exponentially in the number of qubits. This is the **barren plateau** problem, discovered by McClean et al. (2018).

Formally: for random parameterized quantum circuits of sufficient depth, the variance of the cost function's gradient decays as O(1 / 2^n). Once your gradients are indistinguishable from measurement noise, gradient descent stops working.

Known culprits and their fixes:

* **Random deep initialization** causes barren plateaus. Fix: initialize near the identity or use structured circuits like QCNNs.
* **Global observables** (measuring across all qubits) cause plateaus. Fix: use local observables (measure one qubit at a time).
* **Highly expressive ansatze** (able to represent any state) tend to have plateaus. Fix: use restricted ansatze that only cover the relevant subspace.

Below we measure gradient variance for a random ansatz at growing qubit counts to see the plateau in action.

In [ ]:
def cost(theta, n):
    s = ket0(n)
    idx = 0
    for _ in range(3):
        for q in range(n):
            s = apply_1q(Ry(theta[idx]), q, n) @ s
            idx += 1
        for q in range(n-1):
            s = apply_cnot(q, q+1, n) @ s
    # Global observable: <Z tensor Z tensor ... tensor Z>
    Zn = kron_all([Z]*n)
    return float(np.real(np.conj(s) @ Zn @ s))

def grad_at_zero(n, eps=0.01, n_samples=50):
    n_params = 3*n
    gradients = []
    for _ in range(n_samples):
        base = np.random.uniform(0, 2*np.pi, n_params)
        c0 = cost(base, n)
        d = base.copy(); d[0] += eps
        gradients.append((cost(d, n) - c0)/eps)
    return float(np.var(gradients))

for n in [2, 3, 4, 5, 6]:
    v = grad_at_zero(n)
    print(f"n_qubits = {n}: grad variance = {v:.4e}")

### Activity 18.1

Look at the printed variances. Roughly how much does gradient variance decrease when you go from n=2 to n=6? What does that mean for training a large VQC with this design?

In [ ]:
answer_18_1 = """YOUR ANSWER HERE, one sentence."""
print(answer_18_1)

<details><summary>Sample answer</summary>

Variance typically drops by roughly two orders of magnitude going from 2 to 6 qubits, matching the O(1/2^n) prediction. For real hardware with 50 or 100 qubits, gradient signals would be washed out by shot noise, so this design is untrainable without structural fixes like locality restrictions or QCNN-style hierarchies.
</details>

---
# Section 18.2: The Data-Loading Bottleneck

Any QML algorithm claiming a speedup runs into this question: **how did the data get into the quantum state?**

* **Amplitude encoding** packs N features into log2(N) qubits but requires an O(N) state preparation circuit. Any O(N) prep cost erases most exponential speedups downstream.
* **Angle encoding** loads one feature per qubit rotation. Cheap per-feature but requires as many qubits as features.
* **QRAM** would prepare superpositions of data rows in log time, but no scalable QRAM has been built. Assuming QRAM in a paper is like assuming free electricity.
* **Data re-uploading** interleaves data loading with trainable rotations, useful for small inputs. Scales poorly.

The bottleneck: for high-dimensional classical data, loading it onto a quantum register is often as expensive as running the whole classical algorithm.

This is why practical QML today focuses on cases where the data is small (few features), or where the data itself is quantum (from a quantum sensor or simulation), sidestepping the loading problem entirely.

In [ ]:
# Illustrate loading cost: naive amplitude encoding of a random vector using scipy
# We'll count the "number of gates" as a proxy for the circuit depth.
# For a general N-amplitude state, standard results give O(N) gates.
sizes = [4, 8, 16, 32, 64, 128]
gates_naive = [n for n in sizes]  # rough proportional
gates_qram_if_it_existed = [int(np.log2(n)) for n in sizes]

fig, ax = plt.subplots(figsize=(5, 3))
ax.plot(sizes, gates_naive, marker='o', label='amplitude encoding, general')
ax.plot(sizes, gates_qram_if_it_existed, marker='s', label='hypothetical QRAM')
ax.set_xscale('log', base=2); ax.set_yscale('log')
ax.set_xlabel('# amplitudes to load'); ax.set_ylabel('# gates')
ax.set_title('Data loading cost, real vs hypothetical'); ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout(); plt.show()

---
# Section 18.3: When Does QML Actually Beat Classical?

Here is the honest picture as of today:

* **Sensor data straight from a quantum experiment.** No loading cost, quantum in and quantum out. Best current case for near-term QML.
* **Quantum chemistry and materials.** VQE and its descendants are the most credible near-term application. Classical DFT is imperfect, and even modest quantum improvements matter.
* **Provably-fast QML.** Only a handful of examples with rigorous exponential speedups, most requiring FT hardware or unrealistic access models.
* **Sample complexity for certain concept classes.** Recent theoretical work shows QML can learn some functions with exponentially fewer training samples than any classical algorithm.

Where the field is not yet: general image or NLP classification is essentially unlikely to see a real quantum advantage over classical deep learning within the next decade.

---
# Section 18.4: Reading QML Papers and Picking Research Problems

A five-minute triage checklist for any QML paper claiming a speedup:

1. What is the data-access model? QRAM? Streaming? Data re-uploading? If QRAM, treat with extreme skepticism.
2. What is the exact classical baseline? Is it the best-known algorithm or a strawman?
3. What is the barren-plateau story? Is the ansatz structured or random?
4. Has the "speedup" been checked against dequantization arguments (Tang and follow-ups)?
5. Do the resource numbers assume NISQ or FT hardware? If FT, what code distance? What magic state cost?

If any of these are missing, the paper is not yet actionable. Not necessarily wrong, just incomplete.

### Activity 18.2

Pick one of these open problems and write a 3-sentence pitch for how you'd start investigating it:
1. Better data-loading protocols for high-dimensional classical data.
2. Provably plateau-free ansatze for QML on generic tasks.
3. A rigorous separation showing QML beats classical on a well-defined problem.

In [ ]:
pitch_18_2 = """YOUR 3-SENTENCE PITCH HERE."""
print(pitch_18_2)

<details><summary>Sample pitch (data loading)</summary>

I would survey existing near-term encodings (amplitude, angle, re-uploading) and quantify the encoding cost for a benchmark dataset like MNIST. Then I would look at recent "block encoding" methods that trade circuit depth for structure, and see whether any hybrid encoding matches classical performance while keeping the loading cost sub-linear in the data size. The deliverable would be an empirical comparison table on standard benchmarks.
</details>

### Activity 18.3

Reflection: after this course, which open QML problem is most interesting to you personally, and why? (Write a short note; this is for your own learning, not graded.)

In [ ]:
my_reflection = """YOUR REFLECTION HERE."""
print(my_reflection)

---
## Course Wrap-up

This is Unit 18, the final unit of the extended course. Over 18 units you built:

* A quantum simulator from scratch.
* Variational quantum classifiers, quantum kernels, quantum neural networks.
* Quantum algorithms including QFT, QAOA, Grover, HHL, VQE.
* Quantum error correction and fault tolerance.
* QML across NLP, computer vision, recommenders, federated learning, and time series.
* An understanding of quantum advantage benchmarks and open research problems.

You are now equipped to read current QML papers, evaluate speedup claims, and start your own project. Congratulations.

---
## End-of-Unit Quiz (10 multiple choice)

**Q1.** A barren plateau in QML is:

A. A geographic feature
B. A regime where cost-function gradients decay exponentially in qubit count
C. A perfectly flat quantum state
D. A hardware error

**Q2.** The variance of gradients in a random deep ansatz decays as:

A. O(n)
B. O(log n)
C. O(1 / 2^n)
D. Constant

**Q3.** Which of these does NOT help mitigate barren plateaus?

A. Structured ansatze like QCNNs
B. Local observables
C. Deeper random circuits with more parameters
D. Identity initialization

**Q4.** Amplitude encoding of N features requires:

A. log2(N) qubits and O(N) gates for preparation
B. N qubits and log(N) gates
C. 1 qubit total
D. No gates

**Q5.** QRAM in current QML papers is:

A. Widely available hardware
B. An assumed data structure that has not been built at scale
C. A form of classical RAM
D. A benchmark

**Q6.** Which near-term application is currently the strongest for QML?

A. Photo classification on ImageNet
B. Learning from data produced directly by quantum sensors or simulations
C. Real-time video generation
D. Web search

**Q7.** Ewin Tang's dequantization work primarily affects claims that:

A. Rely on structured classical sampling access being unavailable to classical algorithms
B. Use only single-qubit gates
C. Are limited to fault-tolerant hardware
D. Involve chemistry

**Q8.** When triaging a QML paper claiming a speedup, which is a red flag?

A. Reporting resource estimates in both NISQ and FT scenarios
B. Assuming QRAM without discussion
C. Comparing to state-of-the-art classical baselines
D. Reporting statistical error bars

**Q9.** Which classical baseline is the "strawman" version to avoid?

A. Best-known classical algorithm for the exact task
B. A naive brute-force algorithm ignoring standard optimizations
C. A tuned production system
D. A well-cited recent paper

**Q10.** Which of the following is a genuinely open research question in QML?

A. Whether quantum computers can be built
B. Whether QML can beat classical deep learning on generic image tasks in the near term
C. Whether pi is irrational
D. Whether entanglement exists

---
## End-of-unit submission

Fill in your ten multiple choice answers, then run this cell to submit.

In [ ]:
quiz_answers = {
    "q1":  "",   # A, B, C, or D
    "q2":  "",
    "q3":  "",
    "q4":  "",
    "q5":  "",
    "q6":  "",
    "q7":  "",
    "q8":  "",
    "q9":  "",
    "q10": ""
}

reflection = "What did you find most interesting in this unit? (optional)"

_post_event("unit_completed",
            payload={"quiz": quiz_answers, "reflection": reflection})

print(f"Submitted Unit 18!")